In [31]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, WebSearchTool, trace, Runner, function_tool, OpenAIChatCompletionsModel
from agents.model_settings import ModelSettings
import asyncio
import os
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
from datetime import date
# from messenger import send_email, push

In [45]:
load_dotenv(override = True)

True

In [ ]:
model = "gemini-3.8-flash"
use_email = True
number_of_searches = 2
current_date = date.today().isoformat()

In [47]:
task = "Most popular AI Agent frameworks in 2026"

In [48]:
gemini_api_key = os.getenv("GOOGLE_GEMINI_API_KEY")

In [49]:
if gemini_api_key:
    print(f"Geimini API Key exists and begins {gemini_api_key[:4]}")
else:
    print("Geimini API Key not set")


Geimini API Key exists and begins AIza


In [50]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

In [51]:
gemini_client = AsyncOpenAI(base_url = GEMINI_BASE_URL, api_key = gemini_api_key)

In [52]:
gemini_model = OpenAIChatCompletionsModel(model = model, openai_client = gemini_client)

### The Search Agent

In [ ]:
the_search_agent_instructions = f"""
You are a reliable, evidence-driven research assistant in a Deep Research Agent system.Given a search term, you have 
to search the web for that term and produce a concise summary of the most relevant findings.

Prioritize information based on its relevance to the search query, reliability, and usefulness. Consider the current 
date - {current_date}, when assessing information freshness. Prefer recent developments when they materially affect 
the topic, especially in rapidly evolving fields such as Generative AI and Agentic AI, but do not favor newer 
information over older, authoritative information when the latter is more relevant.


Search relevant and reputable sources, prioritizing official documentation, primary sources, peer-reviewed research,
established news organizations, and recognized industry publications as appropriate. Verify important claims against
independent sources when possible. Exclude irrelevant information, unsupported claims, speculation, and redundancy.

Capture and highlight the main points and be succinct.

Return only a factual summary in 2–3 short paragraphs, with a maximum of 300 words. Include relevant source names 
and URLs for downstream verification. Never invent facts or citations; clearly indicate when evidence is insufficient 
or conflicting.


"""

In [55]:
# the_search_agent_instructions

In [56]:
model_settings = ModelSettings(tool_choice = "required")
tools = [WebSearchTool()]

In [57]:
the_search_agent = Agent(
    name = "The Search Agent",
    instructions = the_search_agent_instructions,
    model = gemini_model,
    tools = tools,
    model_settings = model_settings
)

In [58]:
result = await Runner.run(the_search_agent, task)
display(Markdown(result.final_output))

UserError: Hosted tools are not supported with the ChatCompletions API. Got tool type: <class 'agents.tool.WebSearchTool'>, tool: WebSearchTool(user_location=None, filters=None, search_context_size='medium', external_web_access=None, search_content_types=None, image_settings=None)

### The Planner Agent

In [59]:
# Use of Structured Outputs 

In [60]:
class WebSearchItem(BaseModel):

    reason : str = Field(instructions = "Your reasoning for why this search is important for the query")
    search_query : str = Field(instructions = "The search term to use for the web search.")
    

C:\Users\hp\AppData\Local\Temp\ipykernel_21108\2654601362.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'instructions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  reason : str = Field(instructions = "Your reasoning for why this search is important for the query")
C:\Users\hp\AppData\Local\Temp\ipykernel_21108\2654601362.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'instructions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  search_query : str = Field(instructions = "The search term to use for the web search.")


In [61]:
class WebSearchPlan(BaseModel):

    theSearchPlan : list[WebSearchItem] = Field(instructions = "A list of web searches to be performed to answer the query in the best possible way.")
    # "The List of WebSearch Items adhering to the number_of_searches"

C:\Users\hp\AppData\Local\Temp\ipykernel_21108\125153771.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'instructions'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.14/migration/
  theSearchPlan : list[WebSearchItem] = Field(instructions = "A list of web searches to be performed to answer the query in the best possible way.")


In [62]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'instructions': 'Your reasoning for why this search is important for the query',
     'title': 'Reason',
     'type': 'string'},
    'search_query': {'instructions': 'The search term to use for the web search.',
     'title': 'Search Query',
     'type': 'string'}},
   'required': ['reason', 'search_query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'theSearchPlan': {'instructions': 'A list of web searches to be performed to answer the query in the best possible way.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Thesearchplan',
   'type': 'array'}},
 'required': ['theSearchPlan'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [69]:
the_planner_agent_instructions = f"""
You are a research planning assistant. Given a user's research query, generate exactly {number_of_searches} distinct 
and complementary web search queries that together cover the most important aspects of the question and should be performed to 
best answer the query. Focus on relevant information, key evidence, and recent developments where appropriate. Avoid
redundant or overly broad searches.

Return {number_of_searches} queries in the required structured output format.

"""

In [70]:
the_planner_agent = Agent(
    name = "The Planner Agent", 
    instructions= the_planner_agent_instructions,
    model = gemini_model,
    output_type = WebSearchPlan
)

In [71]:
result = await Runner.run(the_planner_agent, task)
print(result.final_output)

theSearchPlan=[WebSearchItem(reason='Identify the leading multi-agent frameworks, recent architectural shifts, and direct feature comparisons among top contenders going into 2025 and 2026.', search_query='top AI agent frameworks comparison 2025 2026 LangGraph CrewAI AutoGen'), WebSearchItem(reason='Gauge actual developer adoption, GitHub momentum, and industry trends for open-source AI agent tooling.', search_query='most popular open source AI agent frameworks adoption benchmarks 2025 2026')]


### The Writer Agent

In [66]:
class ReportData(BaseModel):
    short_summary : str = Field(description = "A short 2-3 sentence summary of the findings.")
    markdown_report : str = Field(description = "The final report")
    follow_up_questions : list[str] = Field(description = "Suggested topics to research further")

In [67]:
instructions= """
You are a senior researcher tasked with writing a cohesive report for a research query. You will be provided with the 
original query, and some research. Generate a comprehensive report based on the research and the query. The final 
output should be in markdown format, and it should be lengthy and detailed. Aim for 5 to 8 pages of content, at most
900 words.
"""

In [68]:
the_writer_agent = Agent(
    name = "The Writer Agent",
    instructions = instructions,
    model = model,
    output_type = ReportData
    )

### The Email Sender Agent

In [ ]:
# def send_email()

In [20]:
@function_tool
def email_sender_tool(subject : str, text_body : str, html_body : str) -> str: 
    """
    This tool will send out an email with the given subject and body to all sales prospects.

    Arguments:
        subject : The subject of the email
        text_body : The body of the email as plain text
        html_body: The HTML body of the email

    """

    if use_email:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")
    return "Email sent sucessfully"


In [22]:
email_sender_tool.params_json_schema

{'properties': {'subject': {'title': 'Subject', 'type': 'string'},
  'text_body': {'title': 'Text Body', 'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'email_sender_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [23]:
instrcuctions = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

In [24]:
email_agent = Agent(
    name = "The Email Sender Agent",
    instrcuctions = instrcuctions
    model = model_name,
    tools = [email_sender_tool]
)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (1440485505.py, line 3)

### The Orchestration by Code

In [ ]:
async def run_searches(query: str):
    print("Planning searches by putting up the Planner Agent to work...")
    results = await Runner.run(the_planner_agent, f"Query : {query}")
    searches = results.final_output.searches
    print("Finished Planning")
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

In [ ]:
async def search(item: WebSearchItem):
    input_message = f"Search term: {item.search_query}\nReason for searching: {item.reason}"
    result = await Runner.run(the_search_agent, input_message)
    return result.final_output

In [ ]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about the report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(the_writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

In [ ]:
async def send_email_report(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

In [ ]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")